# 🏗️ CEM4644 · MP2 — Image classification for construction
## Homework (individual): *crack vs. no crack* and *architectural style*

**No coding needed.** This notebook is a series of buttons. Each grey box below is one *step*: click the ▶ (play) button at its left, wait until it finishes, look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 120 minutes)**
1. Look at labelled photos and try to label some yourself.
2. Run a trained classifier, read its confidence, and measure how often it is right.
3. Try to break it: tricky photos, edited photos, photos from another world, your own photos.
4. Do the same with more than two classes.
5. Invent your own classes and classify with no training at all.
6. Train your own model and compare it with the course model.

**Before you start (optional, makes training faster):** menu *Runtime → Change runtime type → T4 GPU → Save*. Everything also works without a GPU.

**Datasets:** Concrete surface cracks · Architectural styles of building façades (computer-generated images). Sources and licences are listed at the bottom.

In [ ]:
#@title ▶ Step 0 · Run me first (1–2 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the green ✅ line. This downloads the photos and the course models. Nothing else to do here.
#@markdown If Colab asks whether to run a notebook that was not authored by Google, choose *Run anyway*.
import os, sys, subprocess
if not os.path.isdir("CEM4644/mp2_image_classification"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/Haolan-Zhang/CEM4644.git"], check=True)
sys.path.insert(0, os.path.abspath("CEM4644/mp2_image_classification"))
from aec_lab import lab
lab.setup(binary="concrete_cracks", multiclass="facade_styles", task_names={"binary": "crack vs. no crack", "multiclass": "architectural style"}, gradio="6.26.0")


## Part 1 · Meet the data

A classifier learns from **labelled examples**: photos for which a person has already written down the answer. The answer is called the **label**, and each possible answer is a **class** (for example *crack* / *no crack*).

The photos are kept in two separate piles:
- **training photos** — the model learns from these;
- **test photos** — hidden from the model during training, so that we can measure how it does on photos it has never seen. Every accuracy number in this notebook is computed on test photos only.

In [ ]:
#@title ▶ Step 1a · Browse the photos { display-mode: "form" }
#@markdown Choose which task, a class (or *all*), and how many photos. Click ▶ again for a new random selection.
task = "crack vs. no crack" #@param ["crack vs. no crack", "architectural style"]
category = "all" #@param ["all", "no crack", "crack", "Bauhaus International", "Brutalist", "Art Deco", "Neoclassical", "Gothic Revival", "Georgian", "Victorian Terrace", "Mid-century Modern", "Contemporary Curtain Wall", "Industrial Warehouse"]
how_many = 12 #@param {type:"slider", min:4, max:24, step:4}
lab.show_gallery(task, category, how_many)


In [ ]:
#@title ▶ Step 1b · Can *you* tell them apart? { display-mode: "form" }
#@markdown A photo appears: click the class you think is right, then *Next photo*. Your score is shown at the end.
task = "architectural style" #@param ["crack vs. no crack", "architectural style"]
rounds = 8 #@param {type:"slider", min:4, max:20, step:2}
lab.guess_game(task, rounds)


> ### 📝 Report question 1
> What was your score in the guessing game for *architectural style*? Which classes were hard for **you** to tell apart, and what made them hard?

## Part 2 · Run a trained classifier (crack vs. no crack)

The **course model** was trained beforehand on 8,000 labelled photos. Given a new photo it returns a **confidence** for every class (the confidences add up to 100 %) and answers with the class that has the highest confidence.

In [ ]:
#@title ▶ Step 2a · Classify one photo at a time { display-mode: "form" }
#@markdown Click *🎲 Another photo* as often as you like. Watch the confidence bars: is the model always sure? Is it ever confidently wrong?
lab.pick_and_predict("crack vs. no crack")


In [ ]:
#@title ▶ Step 2b · Test it on all the unseen test photos { display-mode: "form" }
#@markdown **Accuracy** = share of test photos classified correctly.
#@markdown The **confusion matrix** shows, for each true class (rows), what the model said (columns). Numbers on the diagonal are correct; everything else is a mistake.
how_many = "all" #@param ["all", "100", "300"]
lab.evaluate("crack vs. no crack", how_many)


> ### 📝 Report question 2
> What accuracy did the course model reach for *crack vs. no crack*, on how many test photos? Would you trust it as the *only* check in a building inspection? Explain in two or three sentences.

In [ ]:
#@title ▶ Step 2c · Look at the mistakes { display-mode: "form" }
#@markdown Use the drop-downs to filter by true class and by what the model said. The most confident mistakes are shown first: those are the interesting ones.
lab.error_explorer("crack vs. no crack")


In [ ]:
#@title ▶ Step 2d · Where do you draw the line? { display-mode: "form" }
#@markdown By default the model says *crack* when its confidence is above 50 %. Move the slider to change that rule and watch the two kinds of mistakes: **missed cracks** and **false alarms**.
lab.threshold_explorer()


> ### 📝 Report question 3
> Move the threshold to 0.1 and to 0.9. What happens to the number of missed *cracks* and the number of false alarms? Which of the two mistakes is worse for a building inspection, and which threshold would you choose? Why?

## Part 3 · Where does it break?

A model only knows the kind of photos it was trained on. Let's look for its limits. In the galleries below the caption says what each photo really is; the model's verdict is printed under it (**green** = agrees with the expected answer, **red** = disagrees, black = there is no single right answer).

In [ ]:
#@title ▶ Step 3a · Tricky photos { display-mode: "form" }
#@markdown *hard_real*: real test photos the model gets wrong · *borderline*: real photos it is barely sure about · *synthetic*: real photos we edited · *other_domain*: photos from a different dataset · *out_of_scope*: not walls at all.
group = "all" #@param ["all", "hard_real", "borderline", "synthetic", "other_domain", "out_of_scope"]
lab.tricky("crack vs. no crack", group)


In [ ]:
#@title ▶ Step 3b · Break it yourself { display-mode: "form" }
#@markdown An app appears below (give it 10–20 seconds; a *public URL* is printed above it, which you can open in a new tab or on your phone).
#@markdown **Sliders tab:** rotate, zoom, blur, darken, cast a shadow, add noise, draw a dark line; the model re-runs on every change. Try to flip its answer with the *smallest* possible change.
#@markdown **Draw on it tab:** paint a crack, a stain or a shadow on the photo with the brush, then click *Classify my drawing*. You can also upload or photograph your own wall and draw on that.
#@markdown **Invisible noise tab:** an *adversarial attack*. The app uses the model's own gradient to compute a pattern of tiny pixel changes (a few steps out of 255, invisible to you) that flips the verdict, then saves the result as a JPEG and checks whether the attack survives. Try fewer steps and lower strength and see when it stops working.
lab.playground_app("crack vs. no crack")


In [ ]:
#@title ▶ Step 3c · A model from a different world { display-mode: "form" }
#@markdown The same test photos are given to a model trained on **Building façade / wall surface defects**. Both models answer 'defect or not', but only one has seen photos like these before.
lab.domain_shift()


In [ ]:
#@title ▶ Step 3d · Your own photo { display-mode: "form" }
#@markdown A small app appears below (also as a public link you can open on your phone to use the camera). Upload or photograph a wall, floor, pavement, a face, a drawing... anything.
#@markdown Test at least 5 photo(s) of your own and take screenshots for your report.
lab.upload_app()


> ### 📝 Report question 4
> List three photos (from the tricky gallery, the sliders, or your own) that fooled the model. For each one, say what the model answered and what you think confused it.

> ### 📝 Report question 5
> What does the model say about a photo that is not a wall at all (the cat, the floor plan, the noise)? Why can it not answer *I don't know*? How would you deal with this if the model were used on a real project?

## Part 4 · More than two classes (architectural style)

Classification is not only about defects. In this part the question is *which architectural style is this façade?* The images are computer-generated reference façades in ten styles, and a course model was trained on them. Keep in mind while you work: none of these images is a real building. The course model for this task was trained on 380 labelled images and chooses between **10 classes**: Bauhaus International, Brutalist, Art Deco, Neoclassical, Gothic Revival, Georgian, Victorian Terrace, Mid-century Modern, Contemporary Curtain Wall, Industrial Warehouse.

In [ ]:
#@title ▶ Step 4a · Classify one image at a time { display-mode: "form" }
#@markdown With many classes the confidence is spread out. Look at the runner-up: is it a *reasonable* second guess?
lab.pick_and_predict("architectural style")


In [ ]:
#@title ▶ Step 4b · Test it on all the unseen test images { display-mode: "form" }
#@markdown Read the confusion matrix row by row: which pairs of classes get mixed up?
lab.evaluate("architectural style", "all")


In [ ]:
#@title ▶ Step 4c · Look at the mistakes { display-mode: "form" }
#@markdown Pick the pair of classes that is confused most often and look at the actual images.
lab.error_explorer("architectural style")


In [ ]:
#@title ▶ Step 4d · Tricky images { display-mode: "form" }
group = "all" #@param ["all", "hard_real", "borderline", "synthetic", "other_domain", "out_of_scope"]
lab.tricky("architectural style", group)


> ### 📝 Report question 6
> For *architectural style*: what is the accuracy, and which two classes are confused most often? Look at a few examples of that confusion. Would a person make the same mistake? Is accuracy alone a fair summary of this model?

## Part 5 · Classes you invent (no training at all)

**CLIP** is a model trained on hundreds of millions of internet photos *together with their captions*. It can compare a photo with any sentence you type. That means you can invent classes on the spot, with no labelled photos: type the class names, and CLIP picks the closest one for each image. This is called **zero-shot** classification.

In [ ]:
#@title ▶ Step 5a · Type your own classes { display-mode: "form" }
#@markdown An app appears below (give it 10–20 seconds; a *public URL* is printed above it). Type class names separated by commas and click **Classify**. The photos stay the same until you click *New photos*, so change the wording and click again to see exactly what your words changed. Try short and descriptive names (*a brick wall*, *a cracked wall*, ...).
#@markdown Second tab: classify your own photo with your own class names. The first click loads CLIP (about a minute).
class_names = "a glass office tower, a stone church, a brick warehouse, a concrete apartment block, a wooden house" #@param {type:"string"}
how_many = 8 #@param {type:"slider", min:4, max:16, step:4}
lab.zero_shot_app(class_names, how_many)


> ### 📝 Report question 7
> Which class names did you try, and did CLIP's answers make sense? Give one construction task where inventing classes like this would be good enough, and one where you would rather train a model on labelled photos. Explain the difference.

## Part 6 · Train your own model

**Training** means showing the model labelled photos, letting it guess, and nudging it a little each time it is wrong. One pass over all the training photos is called an **epoch**.

The course models did not start from zero: they started from a network **pretrained** on 1.2 million everyday photos (ImageNet) and were then **fine-tuned** on our photos. You can start from that pretrained network, or from a **random** network that has never seen a photo in its life.

Suggested experiments (each run takes one to three minutes): 20 photos · 1 pass → 100 photos · 2 passes → 300 photos · 3 passes → then the best setting with a *random* start.

In [ ]:
#@title ▶ Step 6a · Train { display-mode: "form" }
#@markdown Choose the settings, give the run a name, click ▶. Every run is added to the leaderboard in the next step.
task = "crack vs. no crack" #@param ["crack vs. no crack", "architectural style"]
training_photos = "100" #@param ["20", "50", "100", "300", "all"]
passes = 2 #@param {type:"slider", min:1, max:5, step:1}
start = "pretrained" #@param ["pretrained", "random"]
run_name = "run 1" #@param {type:"string"}
n = 10**9 if training_photos == "all" else int(training_photos)
lab.train_my_model(task, n, passes, start, run_name)


In [ ]:
#@title ▶ Step 6b · Leaderboard { display-mode: "form" }
#@markdown All your runs, best first. Copy this table into your report.
lab.leaderboard()


In [ ]:
#@title ▶ Step 6c · Your model vs. the course model on the tricky photos { display-mode: "form" }
#@markdown Each photo shows the verdict of the course model and of your latest model for this task.
lab.compare_my_model("crack vs. no crack")


> ### 📝 Report question 8
> Copy your leaderboard. How did accuracy change with more training photos and more passes? What happened with a *random* start compared with a *pretrained* start, and why do you think that is?

> ### 📝 Report question 9
> Photograph 5 surfaces yourself (walls, floors, pavements, façades) and test them in Step 3d. Include the screenshots. Which verdicts were right? For the wrong ones, what made the photo hard?

> ### 📝 Report question 10
> Name one place in a construction project where a classifier like this could be useful. What photos would you need to collect to train it, who would label them, and what could go wrong?

## Wrap-up

In [ ]:
#@title ▶ Step 7 · Numbers for your report { display-mode: "form" }
#@markdown Prints the accuracies and your training runs in one place.
lab.report_summary()


### Data and model sources
- **Concrete surface cracks** — 227x227 photos of concrete surfaces (floors, walls, columns) of campus buildings, half with a visible crack. Source: Concrete Crack Images for Classification (Özgenel 2019), CC-BY-4.0.
- **Architectural styles of building façades (computer-generated images)** — Computer-generated reference images of building façades in 10 architectural styles, with controlled viewing angle, crop and lighting. None of them shows a real building. Source: Jonathandav/facade-styles, MIT licence.
- **Building façade / wall surface defects** (used in Step 3c) — Close-up photos of concrete and stone walls of more than 50 buildings (10 to 60 years old), taken about 1 m from the wall with a smartphone. Source: BD3 Building Defect Dataset, CC-BY-4.0.
- Course models: ConvNeXt V2 (femto), pretrained on ImageNet-1k by Meta AI (Apache-2.0), fine-tuned for this course.
- Zero-shot model: CLIP ViT-B/32 by OpenAI (MIT).
- Out-of-scope sample images: scikit-image data (public domain / CC0).
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp2_image_classification`).